# Session 1: StampFly とドローン制御の世界
# Session 1: The World of StampFly and Drone Control

## 目標 / Objective

Python SDK で StampFly を操作し、初飛行を体験する。

Experience your first flight by operating StampFly with the Python SDK.

## この Notebook で学ぶこと / What You'll Learn

| トピック | 説明 |
|---------|------|
| SDK 接続 | `connect_or_simulate()` でドローンに接続 |
| 基本コマンド | `takeoff()`, `land()`, `move_forward()` |
| テレメトリ | `get_height()`, `get_battery()`, `get_attitude()` |
| 安全管理 | 緊急停止 `emergency()` の使い方 |

## 1. 環境セットアップ / Environment Setup

まず必要なライブラリをインポートします。

In [ ]:
# Import education helper library
# 教育用ヘルパーライブラリをインポート
from stampfly_edu import connect_or_simulate

import matplotlib.pyplot as plt
import numpy as np

print("Libraries loaded successfully! / ライブラリ読み込み完了！")

## 2. ドローンに接続 / Connect to the Drone

`connect_or_simulate()` は以下の手順で接続を試みます：
1. WiFi 経由で実機 StampFly に接続
2. 接続失敗時はシミュレータモードにフォールバック

シミュレータモードでもすべてのコマンドが動作します。

In [ ]:
# Connect to StampFly (or simulator)
# StampFly に接続（またはシミュレータ）
drone = connect_or_simulate()
print(f"Drone object: {drone}")

## 3. はじめての飛行！ / Your First Flight!

### 理論 / Theory

ドローンの飛行は **フィードバック制御** によって実現されています：

```
目標値 ──→ [コントローラ] ──→ [モーター] ──→ [ドローン] ──→ 実際の状態
  ↑                                                          │
  └──────────────── [センサ] ←─────────────────────────────────┘
```

`takeoff()` を呼ぶと、内部で以下が自動実行されます：
1. モーターが回転開始
2. 高度制御が目標高度（約50cm）を維持
3. 姿勢制御が水平を保つ

### 安全確認 / Safety Check

⚠️ 実機で飛行する前に：
- 周囲に人がいないことを確認
- プロペラガードに破損がないか確認（StampFly は一体型ガード標準搭載）
- バッテリー残量を確認（30%以上推奨）

In [ ]:
# Check battery before flight
# 飛行前にバッテリーを確認
battery = drone.get_battery()
print(f"Battery level: {battery}%")

if battery < 30:
    print("WARNING: Low battery! Charge before flying.")
    print("警告: バッテリー残量低下！充電してから飛行してください。")
else:
    print("Battery OK. Ready to fly! / バッテリーOK。飛行準備完了！")

In [ ]:
# Takeoff!
# 離陸！
drone.takeoff()

# Check current state
# 現在の状態を確認
print(f"Height: {drone.get_height()} cm / 高度: {drone.get_height()} cm")
print(f"Attitude: {drone.get_attitude()} / 姿勢: {drone.get_attitude()}")

In [ ]:
# Move forward 50cm
# 50cm 前進
drone.move_forward(50)
print(f"Position after forward: height={drone.get_height()}cm")

In [ ]:
# Move back to start
# スタート位置に戻る
drone.move_back(50)
print(f"Back to start: height={drone.get_height()}cm")

In [ ]:
# Land
# 着陸
drone.land()
print("Landed safely! / 安全に着陸しました！")

## 4. テレメトリを読む / Reading Telemetry

StampFly は 400Hz でテレメトリデータを送信しています。
`get_telemetry()` で最新データを取得できます。

In [ ]:
# Takeoff and read telemetry
# 離陸してテレメトリを読む
drone.takeoff()

import time

# Collect telemetry for 3 seconds
# 3秒間テレメトリを収集
telemetry_log = []
for i in range(30):  # 30 samples at ~10Hz
    data = drone.get_telemetry()
    data["sample"] = i
    telemetry_log.append(data)
    time.sleep(0.1)

drone.land()

# Display telemetry
# テレメトリを表示
import pandas as pd
df = pd.DataFrame(telemetry_log)
print("Telemetry columns: ", list(df.columns))
df.head(10)

## 5. テレメトリの可視化 / Visualizing Telemetry

収集したテレメトリデータをプロットしてみましょう。

In [ ]:
# Plot altitude during flight
# 飛行中の高度をプロット
if "z" in df.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(df["sample"] * 0.1, df["z"], "b-o", markersize=3)
    ax.set_xlabel("Time (s) / 時間")
    ax.set_ylabel("Altitude (m) / 高度")
    ax.set_title("Altitude during hover / ホバリング中の高度")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No altitude data available in telemetry")

## 6. 考察課題 / Discussion Questions

1. **フィードバック制御**: ドローンが安定してホバリングできるのはなぜですか？もしセンサがなかったらどうなりますか？

2. **テレメトリの揺らぎ**: 高度データが完全にフラットではなく、わずかに揺れているのはなぜですか？

3. **バッテリーと飛行時間**: バッテリー電圧は飛行中にどう変化しますか？これは制御にどう影響しますか？

---

1. **Feedback Control**: Why can the drone hover stably? What would happen without sensors?

2. **Telemetry Fluctuations**: Why isn't the altitude data perfectly flat during hover?

3. **Battery and Flight Time**: How does battery voltage change during flight? How does this affect control?

In [ ]:
# Clean up
# 後片付け
drone.end()
print("Session complete! / セッション完了！")